# Étape 4 : Profiling et Optimisation

In [1]:
import pandas as pd
import numpy as np
import time
import cProfile
import pstats
import io
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import time

# Adapter les imports selon la structure de ton dossier app
import sys
project_root = r"C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset"
model_path = r"C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\models\pmvl_catboost_final.cbm"
features_path = r"C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\models\pmvl_feature_columns.txt"

project_root = r"C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["MODEL_PATH"] = model_path
os.environ["FEATURES_PATH"] = features_path

from app.schemas import PMVLFeatures
from app.main import run_model_prediction, get_model, get_feature_columns


## 1. Profiling (Recherche du goulot d'étranglement)
On utilise `cProfile` pour voir exactement quelles lignes de code prennent le plus de temps dans ta fonction actuelle.

In [2]:
# 1. Profiling détaillé pour trouver le goulot d'étranglement
print("--- PROFILING DÉTAILLÉ DE run_model_prediction ---")

sample_input = {"holding_date": "2026-03-01",
    "pmvl_estim": 1500.0,
    "quantity": 100.0,
    "purch_val_clean": 45000.0,
    "quote": 510.0,
    "vnc_agrege_dirty": 49000.0,
    "entite": "ENTITE_TEST",
    "isin": "FR0000000001",
    "orig_name": "Asset Name Test",
    "ticker": "TICKER_TEST",
    "ref_unik_asset": "REF_12345",
    "fund_code": "FUND_001",
    "col_3a": "3A_TEST",
    "canton": "CANTON_TEST",
    "cic": "CIC_TEST",
    "groupe": "GROUPE_TEST",
    "ptf_name": "PTF_TEST"
}

features = PMVLFeatures(**sample_input)

# Warmup
_ = run_model_prediction(features)

# Profiler
pr = cProfile.Profile()
pr.enable()
for _ in range(50):  # 50 itérations pour avoir des stats stables
    _ = run_model_prediction(features)
pr.disable()

# Afficher les résultats triés par temps cumulé
s = io.StringIO()
sortby = 'cumulative'
ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
ps.print_stats(15)  # Afficher le top 15 des fonctions les plus lentes
print(s.getvalue())

# Hypothèse probable : la création du DataFrame Pandas (pd.DataFrame([row])) 
# ou le make_position_group prend la majorité du temps (plus que le predict_proba de CatBoost).


--- PROFILING DÉTAILLÉ DE run_model_prediction ---
Chargement du modèle CatBoost en mémoire...
Modèle chargé avec succès !
         719178 function calls (703675 primitive calls) in 0.790 seconds

   Ordered by: cumulative time
   List reduced from 592 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    0.788    0.394 c:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3712(run_code)
        2    0.001    0.000    0.788    0.394 {built-in method builtins.exec}
       50    0.003    0.000    0.752    0.015 C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\app\main.py:88(run_model_prediction)
        1    0.005    0.005    0.618    0.618 C:\Users\j-a-b\AppData\Local\Temp\ipykernel_9692\3320419924.py:1(<module>)
       50    0.008    0.000    0.490    0.010 C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\app\main.py:64(pre

## 2. Test d'une stratégie d'optimisation (Code / Software)
L'hypothèse principale est que Pandas est très lent pour manipuler une seule ligne de données. On va réécrire la fonction pour préparer la donnée nativement en Python (liste) avant de la passer à CatBoost, sans créer de DataFrame.

In [3]:
# 2. Version Optimisée de l'inférence

model = get_model()
feature_columns = get_feature_columns()
cat_indices = model.get_cat_feature_indices()
cat_cols = [feature_columns[i] for i in cat_indices]
GROUP_KEYS = ["PMVL[ENTITE]", "PMVL[Selected Fund code]", "PMVL[ISIN]", "PMVL[Ref Unik Asset]"]

def run_model_prediction_optimized(features: PMVLFeatures):
    """
    Version optimisée qui minimise les appels Pandas coûteux pour une seule ligne.
    """
    # 1) Extraire les données brutes
    raw_dict = features.model_dump(by_alias=True)
    raw_dict.pop("PMVL[Holding date]", None)

    # 2) Construire la liste des valeurs dans l'ordre EXACT attendu par CatBoost
    row_values = []

    for col in feature_columns:
        if col == "position_group":
            # Création manuelle de position_group sans Pandas
            parts = [str(raw_dict.get(k, "NA")) for k in GROUP_KEYS]
            if not parts:
                row_values.append("0")
            else:
                row_values.append("||".join(parts))
            continue

        val = raw_dict.get(col)

        # Traitement similaire à prepare_catboost_features mais direct
        if col in cat_cols:
            if val is None or pd.isna(val):
                row_values.append("MISSING")
            else:
                row_values.append(str(val))
        else:
            if val is None:
                row_values.append(np.nan)
            elif isinstance(val, bool):
                row_values.append(int(val))
            else:
                try:
                    row_values.append(float(val))
                except (ValueError, TypeError):
                    row_values.append(np.nan)

    # 3) Prédire directement (CatBoost accepte une liste de listes)
    # C'est BEAUCOUP plus rapide que de créer un DataFrame Pandas
    proba = float(model.predict_proba([row_values])[:, 1][0])

    return proba

# Test de validation (vérifier que l'optimisation donne le même résultat)
proba_standard = run_model_prediction(features).proba_bonne_estimation
proba_opti = run_model_prediction_optimized(features)

print(f"Probabilité Standard: {proba_standard:.6f}")
print(f"Probabilité Optimisée: {proba_opti:.6f}")
print(f"Différence absolue: {abs(proba_standard - proba_opti):.6e}")


Probabilité Standard: 0.031990
Probabilité Optimisée: 0.031990
Différence absolue: 0.000000e+00


## 3. Benchmark final
Comparaison des deux approches.

In [4]:
# 3. Benchmark Comparatif : Standard vs Optimisé
print("--- BENCHMARK COMPARATIF (100 itérations) ---")

# Standard
start = time.perf_counter()
for _ in range(100):
    _ = run_model_prediction(features)
end = time.perf_counter()
time_standard = (end - start) * 1000 / 100

# Optimisé
start = time.perf_counter()
for _ in range(100):
    _ = run_model_prediction_optimized(features)
end = time.perf_counter()
time_opti = (end - start) * 1000 / 100

print(f"Temps moyen Standard : {time_standard:.2f} ms")
print(f"Temps moyen Optimisé : {time_opti:.2f} ms")
gain = (time_standard - time_opti) / time_standard * 100
print(f"Gain de performance : {gain:.1f}%")


--- BENCHMARK COMPARATIF (100 itérations) ---
Temps moyen Standard : 9.06 ms
Temps moyen Optimisé : 0.59 ms
Gain de performance : 93.4%
